# Smoke Test — HotpotQA Multi-Hop Pipeline
5 HotpotQA dev examples, real local LLM, no plots. Mirrors `00_smoke_test.ipynb` but exercises the dataset-aware path (`load_hotpotqa` → `poison_hotpotqa` → `qa_scorer`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import yaml
from pathlib import Path

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

N_SMOKE           = 5
SMOKE_POISON_RATE = 0.5
MODEL             = cfg['models']['default']
PROMPT_TYPE       = cfg['prompts'].get('default_qa', 'standard_qa')
TEMPERATURE       = cfg['models']['temperature']

print(f'model={MODEL}  prompt={PROMPT_TYPE}  n_smoke={N_SMOKE}  poison_rate={SMOKE_POISON_RATE}')

model=Qwen/Qwen2.5-1.5B-Instruct  prompt=standard_qa  n_smoke=5  poison_rate=0.5


In [2]:
# Idempotent download — first run fetches ~46 MB; subsequent runs are no-ops.
from src.data.download_hotpotqa import download

dev_path = Path('..') / cfg['dataset']['hotpotqa_dev']
download(target=dev_path)
print(f'HotpotQA dev available at: {dev_path}')

HotpotQA dev available at: ../data/hotpotqa/dev.jsonl


In [3]:
from src.data.hotpotqa_loader import load_hotpotqa

examples = load_hotpotqa(str(dev_path), max_examples=N_SMOKE)
print(f'Loaded {len(examples)} examples')
print('Keys:', list(examples[0].keys()))
print('Questions:')
for ex in examples:
    print(f'  - {ex["question"]!r}  ->  {ex["answer"]!r}')

Loaded 5 examples
Keys: ['question', 'answer', 'supporting_facts', 'context']
Questions:
  - 'Were Scott Derrickson and Ed Wood of the same nationality?'  ->  'yes'
  - 'What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?'  ->  'Chief of Protocol'
  - 'What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?'  ->  'Animorphs'
  - 'Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?'  ->  'no'
  - 'The director of the romantic comedy "Big Stone Gap" is based in what New York city?'  ->  'Greenwich Village, New York City'


In [4]:
from src.data.hotpotqa_poisoner import poison_hotpotqa

poisoned = poison_hotpotqa(examples, poison_rate=SMOKE_POISON_RATE, seed=cfg['seed'])
n_with_poison = sum(1 for ex in poisoned if ex.get('poisoned_positions'))
print(f'Poisoned {n_with_poison}/{len(poisoned)} examples at rate {SMOKE_POISON_RATE}')

Poisoned 4/5 examples at rate 0.5


In [5]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever
from src.generation.llm_client import HuggingFaceClient

emb_cache = os.path.join('../' + cfg['cache']['dir'], cfg['cache']['embeddings_subdir'])
embedder = Embedder(model_name=cfg['retrieval']['embedding_model'], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg['retrieval']['k'])

llm_cache = os.path.join('../', cfg['cache']['dir'], cfg['cache']['llm_subdir'])
llm = HuggingFaceClient(model=MODEL, temperature=TEMPERATURE, cache_dir=llm_cache)
print(f'Embedder: {cfg["retrieval"]["embedding_model"]} | LLM: {MODEL}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder: all-MiniLM-L6-v2 | LLM: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
from src.evaluation import qa_scorer

with embedder, llm:
    metrics_clean = qa_scorer.run(
        examples=examples,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        max_tokens_by_prompt=cfg['prompts']['max_tokens'],
        seed=cfg['seed'],
        self_consistency_runs=1,
    )

print('\n=== HotpotQA Smoke (poison_rate=0.0) ===')
for k, v in metrics_clean.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== HotpotQA Smoke (poison_rate=0.0) ===
  exact_match: 0.4000
  token_f1: 0.5500
  precision_at_k: 0.3200


In [7]:
with embedder, llm:
    metrics_poisoned = qa_scorer.run(
        examples=poisoned,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        max_tokens_by_prompt=cfg['prompts']['max_tokens'],
        seed=cfg['seed'],
        self_consistency_runs=1,
    )

print(f'\n=== HotpotQA Smoke (poison_rate={SMOKE_POISON_RATE}) ===')
for k, v in metrics_poisoned.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print('\nHotpotQA pipeline wired correctly.' if metrics_poisoned else 'Check errors above.')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== HotpotQA Smoke (poison_rate=0.5) ===
  exact_match: 0.2000
  token_f1: 0.3500
  precision_at_k: 0.2000

HotpotQA pipeline wired correctly.
